# Session 4 — Your turn: Evaluation: when is a model good?

The session chose a threshold under the policy "find at least 95% of the malignant cases". Here the written policy is different, and you will apply the protocol again:
tune and choose on the validation set, lock, then open the test set **once**.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

data = load_breast_cancer(as_frame=True)
X = data.data
y = (data.target == 0).astype(int)              # 1 = malignant, the case we want to find
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.40, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=2000)).fit(X_train, y_train)
val_probability = model.predict_proba(X_val)[:, 1]
test_probability = model.predict_proba(X_test)[:, 1]          # do not look at this until task 1 tells you to
print(len(X_train), len(X_val), len(X_test))
from aij_check import check, close, summary

341 114 114


## Task 1: A different policy

New policy: **precision of at least 0.99 on the validation set (almost no false alarms), and among the thresholds that satisfy it, the highest recall.**
Try the thresholds `np.linspace(0.05, 0.95, 91)` on the validation set, store the best one in `threshold`, its validation precision and recall in `val_precision` and `val_recall`.
Only then open the test set once at that threshold: `test_precision` and `test_recall`.

In [2]:
thresholds = np.linspace(0.05, 0.95, 91)
best = None
for t in thresholds:
    predicted = (val_probability >= t).astype(int)
    if predicted.sum() == 0: continue
    p, r = precision_score(y_val, predicted), recall_score(y_val, predicted)
    if p >= 0.99 and (best is None or r > best[2]): best = (t, p, r)
threshold, val_precision, val_recall = best
test_predicted = (test_probability >= threshold).astype(int)
test_precision, test_recall = precision_score(y_test, test_predicted), recall_score(y_test, test_predicted)
print(threshold, val_precision, val_recall, test_precision, test_recall)

0.25999999999999995 1.0 1.0 0.975609756097561 0.9523809523809523


In [3]:
check("the threshold meets the policy on the validation set", lambda: precision_score(y_val, (val_probability >= threshold).astype(int)) >= 0.99,
      "Precision on the VALIDATION set must be at least 0.99.")
check("no other threshold meets the policy with a higher recall", lambda: all(
    recall_score(y_val, (val_probability >= t).astype(int)) <= val_recall + 1e-9 for t in np.linspace(0.05, 0.95, 91)
    if (val_probability >= t).sum() > 0 and precision_score(y_val, (val_probability >= t).astype(int)) >= 0.99))
check("the validation numbers match the threshold", lambda: close(val_recall, recall_score(y_val, (val_probability >= threshold).astype(int)), 1e-9))
check("the test numbers were measured at that threshold", lambda: close(test_recall, recall_score(y_test, (test_probability >= threshold).astype(int)), 1e-9)
      and close(test_precision, precision_score(y_test, (test_probability >= threshold).astype(int)), 1e-9))
summary()

PASS  the threshold meets the policy on the validation set


PASS  no other threshold meets the policy with a higher recall
PASS  the validation numbers match the threshold
PASS  the test numbers were measured at that threshold

4 of 4 checks passed.  Well done.


**Why it works.** A written policy turns 'a good threshold' into a search with a stated goal. The validation set is spent on the choice; the test set is opened once afterwards, so its numbers are an honest estimate. Notice that the test precision can fall short of the validation precision: 0.99 was a target on 43 malignant cases, not a promise.

## Task 2: The four boxes by hand

A model flagged 40 tumours as malignant. Of them 38 were malignant (true positives) and 2 were benign (false positives). It missed 4 malignant tumours (false negatives) and correctly cleared 70 benign ones (true negatives).
Compute, **without scikit-learn**, `precision`, `recall`, `f1` and `accuracy`, and the number of malignant cases missed per 1,000 malignant cases, `missed_per_1000`.

In [4]:
tp, fp, fn, tn = 38, 2, 4, 70
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
accuracy = (tp + tn) / (tp + fp + fn + tn)
missed_per_1000 = 1000 * fn / (tp + fn)
print(precision, recall, f1, accuracy, missed_per_1000)

0.95 0.9047619047619048 0.9268292682926829 0.9473684210526315 95.23809523809524


In [5]:
check("precision = TP / (TP + FP)", lambda: close(precision, 38 / 40, 1e-9))
check("recall = TP / (TP + FN)", lambda: close(recall, 38 / 42, 1e-9))
check("f1 is the harmonic mean of the two", lambda: close(f1, 2 * (38 / 40) * (38 / 42) / ((38 / 40) + (38 / 42)), 1e-9))
check("accuracy counts both kinds of correct answer", lambda: close(accuracy, 108 / 114, 1e-9))
check("about 95 malignant cases missed per 1,000", lambda: close(missed_per_1000, 95.238, 0.01))
summary()

PASS  precision = TP / (TP + FP)
PASS  recall = TP / (TP + FN)
PASS  f1 is the harmonic mean of the two
PASS  accuracy counts both kinds of correct answer
PASS  about 95 malignant cases missed per 1,000

5 of 5 checks passed.  Well done.


**Why it works.** Accuracy (0.947) looks better than recall (0.905) because the 70 easy benign cases pull it up. Which number matters depends on which mistake costs more, which is why the session asks for a written policy before any plot.